# 02 · C++: an autograd engine, Adam and checkpoints that resume exactly

*Base Model Training in C, C++ and CUDA*, notebook 2 of 4

In notebook 1 you put the backward calls in order yourself, so every change to the architecture meant rewriting `compute_gradients`. Frameworks avoid that by **recording** the forward pass as a graph and deriving the backward pass from the record. In this notebook you build such an engine in C++: the same design as the NumPy engine of *Base Model Training*, with memory that frees itself through C++'s ownership rules. Then you add Adam, a training step and a binary checkpoint format that resumes training exactly, and you drive the engine from Python through a small C interface.

## What you will build

- `topological_order(root)`: every node of a graph, parents before children;
- `backward(root)`: reverse-mode differentiation over that order, with gradients that accumulate in the parameters;
- `matmul`: a complete operation, its forward loop and its backward rule written as a C++ lambda;
- `cross_entropy`: a fused, numerically stable log-softmax and loss with its rule;
- `Adam::step` and `train_step`: an optimiser and one correct training step;
- `load_checkpoint`: a reader for a binary checkpoint format that rejects damaged files.

## Learning objectives

By the end of this notebook you will be able to:

1. explain RAII, `std::vector` and `std::shared_ptr` ownership, and why a graph of shared pointers frees itself unless it contains a cycle;
2. record a computation as a graph of nodes whose backward rules are lambdas, and run reverse-mode differentiation over a topological order;
3. write the forward and backward rules of matrix multiplication and of a fused softmax cross-entropy, and check them numerically;
4. implement Adam with bias correction and a correct training step;
5. design a binary checkpoint format, validate it when loading, and prove that training resumes exactly;
6. expose C++ to Python through an `extern "C"` interface that never lets an exception escape.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you write C++ inside the string that a small Python function returns: delete the line `raise NotImplementedError`, type your code where the comment says, and run the check cell beneath it. The check compiles the engine with your code into a small test program, runs it in a separate process and compares what it prints with what the reference solution prints:

- ✅ means your code matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not compile or does not match yet. The check shows the compiler's message or the first difference. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

C++ compilers can print long messages, especially about templates. Read the **first** error: it names your function, as in `your matmul:9:26`, and the lines after it are usually consequences of that one mistake.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0 for Python:** the standard library only, as in notebook 1. `ctypes` loads the compiled engine, `subprocess` runs the compiler and the check programs, and `array` holds the training data in C memory. NumPy is not used.
- **A C++17 compiler:** g++ or clang++. The course notebook server and Google Colab have one; on your own computer, Linux and macOS (with the Xcode command-line tools) work, and on Windows, use WSL. The engine uses only the C++ standard library.
- **Not in the browser preview:** JupyterLite cannot run a compiler.
- Nothing is downloaded. Checkpoints are written to a temporary folder that is removed when the kernel stops.
- You should have finished notebook 1, or know the backward pass of a dense network. The C++ features used here (classes, references, `std::vector`, `std::shared_ptr` and lambdas) are explained as they appear.

## Setup

Run this cell first. It defines the same tools as notebook 1: `build`, `load` and `run` compile and run code, `mark` labels code for compiler messages, and `practice_native` and `practice` check what you write.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import atexit
import ctypes
import hashlib
import math
import os
import random
import shutil
import signal
import subprocess
import sys
import tempfile
from array import array

if sys.platform == "emscripten":
    raise RuntimeError(
        "This notebook compiles C, C++ and CUDA, which a web browser cannot do. Open it on your "
        "course notebook server, in Google Colab, or in Jupyter on a computer with a C compiler."
    )

CC = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
CXX = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
if not (CC and CXX):
    raise RuntimeError("No C and C++ compiler found: install gcc and g++ (or clang), then restart the kernel.")
BUILD_DIR = tempfile.mkdtemp(prefix="furina-native-")
atexit.register(shutil.rmtree, BUILD_DIR, True)
COMPILED = {}  # source hash -> path of the program or library built from it


class CompileError(Exception):
    """The compiler rejected the code. The message is what the compiler printed."""


class RunError(Exception):
    """A compiled program crashed, never finished, or stopped with an error."""


def _trim(text, lines=14):
    kept = text.strip().splitlines()
    return "\n".join(kept[:lines] + (["   ..."] if len(kept) > lines else []))


def mark(source, label):
    """Splice in source so that compiler messages name `label` and count from its first line."""
    return f'\n#line 1 "{label}"\n{source}\n'


def build(source, language="c", shared=False, flags=()):
    """Compile C ("c"), C++ ("c++") or CUDA ("cuda") into a program, or a shared library."""
    key = hashlib.sha256(f"{language}|{shared}|{flags}|{source}".encode()).hexdigest()[:24]
    if key in COMPILED:
        return COMPILED[key]
    output = os.path.join(BUILD_DIR, key + (".so" if shared else ".exe"))
    if language == "cuda":
        command = cuda_command(source, key, output)  # defined in the CUDA notebooks
    else:
        path = os.path.join(BUILD_DIR, key + (".c" if language == "c" else ".cpp"))
        with open(path, "w", encoding="utf-8") as handle:
            handle.write(source)
        if language == "c":
            command = [CC, "-std=c11", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output, "-lm"]
        else:
            command = [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output]
    if shared:
        command[1:1] = ["-shared"] + (["-Xcompiler", "-fPIC"] if command[0] == globals().get("NVCC") else ["-fPIC"])
    command += list(flags)  # last, so that a flag such as -O1 overrides the default
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise CompileError(_trim(result.stderr or result.stdout))
    COMPILED[key] = output
    return output


def load(source, language="c"):
    """Compile source into a shared library and load it into this Python process."""
    return ctypes.CDLL(build(source, language, shared=True))


def run(program, timeout=20, env=None):
    """Run a compiled program in its own process and return what it printed."""
    try:
        result = subprocess.run([program], capture_output=True, text=True, timeout=timeout,
                                env=None if env is None else {**os.environ, **env})
    except subprocess.TimeoutExpired:
        raise RunError(f"did not finish within {timeout} seconds: is there a loop that never ends?") from None
    if result.returncode < 0:
        try:
            name = signal.Signals(-result.returncode).name
        except ValueError:
            name = f"signal {-result.returncode}"
        hint = {
            "SIGSEGV": "a read or write outside memory the program owns, such as an index past the end of an array",
            "SIGBUS": "a read or write outside memory the program owns",
            "SIGFPE": "an integer division by zero",
            "SIGABRT": "the program aborted itself, often after finding corrupted memory",
            "SIGTRAP": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
            "SIGILL": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
        }.get(name, "")
        detail = f"\n{_trim(result.stderr)}" if result.stderr.strip() else ""
        raise RunError(f"crashed with {name}" + (f" ({hint})" if hint else "") + detail)
    if result.returncode != 0:
        raise RunError(f"stopped with exit status {result.returncode}:\n{_trim(result.stderr + result.stdout)}")
    return result.stdout


def _value(token):
    try:
        return float(token)
    except ValueError:
        return None


def _same(a, b, atol, rtol):
    if a == b:
        return True
    x, y = _value(a), _value(b)
    if x is None or y is None:
        return False
    if math.isnan(x) or math.isnan(y):
        return math.isnan(x) and math.isnan(y)
    return math.isclose(x, y, rel_tol=rtol, abs_tol=atol)


def _short(text, limit=110):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def compare_output(got, want, atol=1e-9, rtol=1e-9):
    """None when two printouts match line by line, otherwise the first difference."""
    got_lines, want_lines = got.strip().splitlines(), want.strip().splitlines()
    for got_line, want_line in zip(got_lines, want_lines):
        got_label, _, got_values = got_line.partition(":")
        want_label, _, want_values = want_line.partition(":")
        g, w = got_values.split(), want_values.split()
        if got_label != want_label or len(g) != len(w) or not all(_same(a, b, atol, rtol) for a, b in zip(g, w)):
            return (f"for {want_label.strip() or 'a case'}\n   your code printed {_short(got_values) or '(nothing)'}"
                    f"\n   expected          {_short(want_values)}")
    if len(got_lines) < len(want_lines):
        return f"your program stopped before printing {want_lines[len(got_lines)].partition(':')[0]!r}"
    if len(got_lines) > len(want_lines):
        return f"your program printed {len(got_lines) - len(want_lines)} line(s) more than expected"
    return None


REFERENCE_OUTPUT = {}


def practice_native(name, yours, reference, program, language="c", atol=1e-9, rtol=1e-9, timeout=20):
    """Check native code you wrote against the reference.

    `yours` and `reference` return source code. `program(source)` wraps a
    source in a complete program whose main() runs the cases and prints the
    results. Both programs are compiled and run in their own processes, so a
    crash or an endless loop fails the check without stopping this notebook.
    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference, so you can always keep going.
    """
    fallback = "   The rest of the notebook uses the reference version for now."
    key = hashlib.sha256(f"{language}|{program(reference())}".encode()).hexdigest()
    if key not in REFERENCE_OUTPUT:
        REFERENCE_OUTPUT[key] = run(build(program(reference()), language), timeout)
    try:
        source = yours()
        if not isinstance(source, str):
            raise TypeError(f"{name} must return its source code as a string, not {type(source).__name__}")
        got = run(build(program(source), language), timeout)
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except CompileError as error:
        print(f"❌ {name}: the compiler rejected your code:\n{error}")
        print(fallback)
        return reference
    except RunError as error:
        print(f"❌ {name}: the check program {error}")
        print(fallback)
        return reference
    except Exception as error:  # a mistake in the Python around the source
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    difference = compare_output(got, REFERENCE_OUTPUT[key], atol, rtol)
    if difference:
        print(f"❌ {name}: {difference}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def _close(got, want, atol):
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want))
    if isinstance(want, float) or isinstance(got, float):
        try:
            return math.isclose(float(got), float(want), rel_tol=1e-9, abs_tol=atol)
        except (TypeError, ValueError):
            return False
    return got == want


def practice(name, yours, reference, cases, atol=1e-9):
    """Check a Python function you wrote against the reference on a few inputs."""
    fallback = "   The rest of the notebook uses the reference version for now."
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(repr(args))}")
                print(f"   you returned {_short(repr(got))}")
                print(f"   expected     {_short(repr(want))}")
                print(fallback)
                return reference
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except Exception as error:
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def address(buffer):
    """The memory address of an array.array, to pass to C as a pointer."""
    return buffer.buffer_info()[0]


def c_list(values, kind="double"):
    """Python numbers as a C initialiser list, such as {1.5, -2.0}."""
    if kind == "int":
        return "{" + ", ".join(str(int(v)) for v in values) + "}"
    suffix = "f" if kind == "float" else ""
    return "{" + ", ".join(f"{float(v)!r}{suffix}" for v in values) + "}"


print(f"C compiler: {CC}\nC++ compiler: {CXX}")

## 1 · Ownership: memory that frees itself

In C, every `malloc` needs a matching `free`, and forgetting one leaks memory while freeing one twice corrupts it. C++ ties memory to objects instead. This idea is called **RAII**, for *resource acquisition is initialisation*: an object acquires a resource when it is created, and its **destructor** releases it automatically when the object goes away, at the end of its scope or when its owner is destroyed.

- `std::vector<double>` owns a growable block of doubles; the block is freed when the vector is destroyed. `v.data()` is the same raw `double *` that notebook 1 passed around, and `v[i]` reads element `i`.
- `std::shared_ptr<Node>` is a pointer that **counts its owners**. Copying it adds an owner, destroying a copy removes one, and when the count reaches zero the `Node` is destroyed. `std::make_shared<Node>(...)` creates a node with one owner.

The engine is a graph of `Node`s. Each node owns its **parents**, the nodes it was computed from, through shared pointers. A loss therefore keeps its whole graph alive, back to the inputs and the weights, and when the last pointer to the loss goes away the graph is destroyed node by node, with no `free` anywhere. A counter, `live_nodes`, goes up in the constructor and down in the destructor, so leaks become visible.

Each node also holds a **backward rule**: a `std::function<void(Node&)>`, a value that holds any callable, here a lambda. During the backward pass the engine calls the rule with the node itself; the rule reads the node's `grad` and **adds** into its parents' gradients. Read the core below.

In [ ]:
CORE_NODES = r"""
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <functional>
#include <memory>
#include <stdexcept>
#include <string>
#include <unordered_set>
#include <vector>

struct Node;
using Tensor = std::shared_ptr<Node>;   // shared ownership: a node lives while anything points to it

static long live_nodes = 0;             // how many nodes exist right now, so that leaks show

struct Node {
    int rows, cols;
    std::vector<double> data;                  // row-major values
    std::vector<double> grad;                  // empty until a gradient arrives, then data's size
    std::vector<Tensor> parents;               // the inputs; owning them keeps them alive
    std::function<void(Node&)> backward_rule;  // reads this node's grad, adds into the parents' grads
    bool requires_grad;
    const char* op;

    Node(int r, int c, bool needs_grad, const char* name)
        : rows(r), cols(c), data(static_cast<size_t>(r) * c, 0.0), requires_grad(needs_grad), op(name) { ++live_nodes; }
    ~Node() { --live_nodes; }
    Node(const Node&) = delete;                // a node is never copied, only shared
    Node& operator=(const Node&) = delete;
    size_t size() const { return data.size(); }
};

Tensor tensor(int rows, int cols, const std::vector<double>& values, bool requires_grad = false)
{
    if (values.size() != static_cast<size_t>(rows) * cols)
        throw std::invalid_argument("tensor: the number of values does not match the shape");
    Tensor t = std::make_shared<Node>(rows, cols, requires_grad, "leaf");
    t->data = values;
    return t;
}

// The output of an operation. It needs a gradient when any input does; only
// then does it keep its inputs, because a constant needs no history.
Tensor result(int rows, int cols, std::vector<Tensor> parents, const char* op)
{
    bool needs = false;
    for (const Tensor& parent : parents)
        needs = needs || parent->requires_grad;
    Tensor out = std::make_shared<Node>(rows, cols, needs, op);
    if (needs)
        out->parents = std::move(parents);
    return out;
}

// A node's gradient buffer, created as zeros the first time a gradient arrives.
std::vector<double>& grad_of(Node& node)
{
    if (node.grad.empty())
        node.grad.assign(node.size(), 0.0);
    return node.grad;
}

"""

`result` creates the output of an operation, and keeps the parents only when some parent needs a gradient: constants carry no history. `grad_of` creates a node's gradient buffer as zeros the first time a gradient arrives. The operations below follow one pattern, which you will use for `matmul`: compute the output's data, then, if the output needs a gradient, attach a rule that adds the local derivative times the upstream gradient `self.grad` into each parent that needs one. A rule may capture plain values such as sizes, but it never captures the output's own `shared_ptr`; the next section shows why.

In [ ]:
CORE_OPS = r"""void same_shape(const Tensor& a, const Tensor& b, const char* op)
{
    if (a->rows != b->rows || a->cols != b->cols)
        throw std::invalid_argument(std::string(op) + ": the shapes differ");
}

Tensor add(const Tensor& a, const Tensor& b)
{
    same_shape(a, b, "add");
    Tensor out = result(a->rows, a->cols, {a, b}, "add");
    for (size_t i = 0; i < a->size(); ++i)
        out->data[i] = a->data[i] + b->data[i];
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            for (const Tensor& parent : self.parents)
                if (parent->requires_grad) {
                    std::vector<double>& g = grad_of(*parent);
                    for (size_t i = 0; i < g.size(); ++i)
                        g[i] += self.grad[i];
                }
        };
    return out;
}

Tensor mul(const Tensor& a, const Tensor& b)
{
    same_shape(a, b, "mul");
    Tensor out = result(a->rows, a->cols, {a, b}, "mul");
    for (size_t i = 0; i < a->size(); ++i)
        out->data[i] = a->data[i] * b->data[i];
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            Node& a = *self.parents[0];
            Node& b = *self.parents[1];
            if (a.requires_grad) {
                std::vector<double>& g = grad_of(a);
                for (size_t i = 0; i < g.size(); ++i)
                    g[i] += self.grad[i] * b.data[i];
            }
            if (b.requires_grad) {
                std::vector<double>& g = grad_of(b);
                for (size_t i = 0; i < g.size(); ++i)
                    g[i] += self.grad[i] * a.data[i];
            }
        };
    return out;
}

Tensor add_bias(const Tensor& x, const Tensor& b)     // b is 1 x cols, added to every row
{
    if (b->rows != 1 || b->cols != x->cols)
        throw std::invalid_argument("add_bias: the bias must be one row as wide as x");
    Tensor out = result(x->rows, x->cols, {x, b}, "add_bias");
    for (int i = 0; i < x->rows; ++i)
        for (int j = 0; j < x->cols; ++j)
            out->data[i * x->cols + j] = x->data[i * x->cols + j] + b->data[j];
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            Node& x = *self.parents[0];
            Node& b = *self.parents[1];
            if (x.requires_grad) {
                std::vector<double>& g = grad_of(x);
                for (size_t i = 0; i < g.size(); ++i)
                    g[i] += self.grad[i];
            }
            if (b.requires_grad) {
                std::vector<double>& g = grad_of(b);
                for (int i = 0; i < self.rows; ++i)
                    for (int j = 0; j < self.cols; ++j)
                        g[j] += self.grad[i * self.cols + j];
            }
        };
    return out;
}

Tensor relu(const Tensor& x)
{
    Tensor out = result(x->rows, x->cols, {x}, "relu");
    for (size_t i = 0; i < x->size(); ++i)
        out->data[i] = x->data[i] > 0.0 ? x->data[i] : 0.0;
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            Node& x = *self.parents[0];
            std::vector<double>& g = grad_of(x);
            for (size_t i = 0; i < g.size(); ++i)
                if (x.data[i] > 0.0)
                    g[i] += self.grad[i];
        };
    return out;
}

Tensor sum(const Tensor& x)                           // every entry added up, as a 1 x 1 tensor
{
    Tensor out = result(1, 1, {x}, "sum");
    double total = 0.0;
    for (double v : x->data)
        total += v;
    out->data[0] = total;
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            for (double& g : grad_of(*self.parents[0]))
                g += self.grad[0];
        };
    return out;
}
"""

ENGINE_CORE = CORE_NODES + CORE_OPS

The check programs use two helpers: `show`, which prints a vector on one labelled line with 17 significant digits, and `numbers`, which produces the same test numbers on every machine. `engine(*pieces)` assembles the core and a list of `(label, source)` pieces into one program, so that each piece keeps its own name in compiler messages.

The cell after it builds a tiny graph and prints how many owners each tensor has and how many nodes are alive.

In [ ]:
CHECK_CPP = r"""
#include <unordered_map>

static void show(const char* label, const std::vector<double>& values)
{
    std::printf("%s:", label);
    for (double v : values)
        std::printf(" %.17g", v);
    std::printf("\n");
}

static std::vector<double> numbers(int count, double seed)   // deterministic test numbers
{
    std::vector<double> values(count);
    for (int i = 0; i < count; ++i)
        values[i] = std::sin(seed + 0.37 * i);
    return values;
}
"""


def engine(*pieces):
    """The engine's source: the given core, then each (label, source) piece in order."""
    return ENGINE_CORE + "".join(mark(source, label) for label, source in pieces)


def checked_program(pieces, main):
    """A check program: the engine with these pieces, the check helpers, then main()."""
    return engine(*pieces) + mark(CHECK_CPP + main, "the check")

In [ ]:
OWNERSHIP_DEMO = r"""
int main()
{
    std::printf("live nodes at the start: %ld\n", live_nodes);
    {
        Tensor a = tensor(1, 2, {1.0, 2.0}, true);
        std::printf("owners of a: %ld\n", a.use_count());
        Tensor s = add(a, a);                                  // s owns a twice, once per parent slot
        std::printf("owners of a after s = add(a, a): %ld\n", a.use_count());
        Tensor total = sum(s);
        std::printf("total = %g, live nodes: %ld\n", total->data[0], live_nodes);
    }                                                          // a, s and total go out of scope here
    std::printf("live nodes after the scope ends: %ld\n", live_nodes);
    return 0;
}
"""
print(run(build(engine() + OWNERSHIP_DEMO, "c++")))

**What to notice.** `a` has one owner, then three: the variable and both parent slots of `s`. The graph holds three nodes, and when the braces close all three are destroyed in the right order, so `live_nodes` returns to 0. Nothing called `free`.

## 2 · The order of the backward pass

A node's backward rule may run only after its own gradient is complete, which means after every node that used it has run its rule. Visiting the nodes in reverse **topological order** guarantees that: a topological order lists every node after all of its parents, so walking it backwards visits each node after every node that used it.

A depth-first search produces such an order: visit a node's parents first, then append the node. A set of the nodes already visited makes sure each node appears once, even when two paths reach it. In C++:

- `std::unordered_set<Node*> seen` is a hash set of raw pointers. `seen.insert(node)` returns a pair whose `.second` is `false` when the node was already present.
- `parent.get()` gives the raw `Node *` inside a `shared_ptr`. Raw pointers are fine here: they are used only during the call, while the graph's shared pointers keep every node alive.
- `const Tensor&` passes a shared pointer by reference, without copying it, so no owner is added.

### ✍️ Your turn: `visit` and `topological_order`

Complete two functions:

- `visit(node, seen, order)`: if `node` is already in `seen`, return. Otherwise add it, call `visit` on each of its parents, then append `node` to `order`;
- `topological_order(root)`: create an empty `order` and `seen`, call `visit(root.get(), seen, order)` and return `order`.

The root comes last, and every node comes after all of its parents. Several orders are correct for most graphs, so the check reports four facts about yours instead of comparing lists: the number of nodes, whether every node comes after its parents, whether no node repeats, and whether the root is last.

*Type it rather than pasting it: typing is how the syntax sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def topological_order_cpp_reference():
    return r"""
void visit(Node* node, std::unordered_set<Node*>& seen, std::vector<Node*>& order)
{
    if (!seen.insert(node).second)          // insert reports false when node was already there
        return;
    for (const Tensor& parent : node->parents)
        visit(parent.get(), seen, order);
    order.push_back(node);
}

std::vector<Node*> topological_order(const Tensor& root)
{
    std::vector<Node*> order;
    std::unordered_set<Node*> seen;
    visit(root.get(), seen, order);
    return order;
}
"""

In [ ]:
def topological_order_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete both functions below
    return r"""
void visit(Node* node, std::unordered_set<Node*>& seen, std::vector<Node*>& order)
{
    /* ✍️ Your code here (about five lines) */
}

std::vector<Node*> topological_order(const Tensor& root)
{
    /* ✍️ Your code here (about four lines) */
}
"""

In [ ]:
TOPOLOGY_MAIN = r"""
static void report(const char* label, const Tensor& root)
{
    std::vector<Node*> order = topological_order(root);
    std::unordered_map<Node*, size_t> position;
    for (size_t i = 0; i < order.size(); ++i)
        position.emplace(order[i], i);
    bool after_parents = true;
    for (Node* node : order)
        for (const Tensor& parent : node->parents)
            if (!position.count(parent.get()) || position[parent.get()] >= position[node])
                after_parents = false;
    bool no_repeats = position.size() == order.size();
    bool root_last = !order.empty() && order.back() == root.get();
    std::printf("%s: %zu %d %d %d\n", label, order.size(), after_parents, no_repeats, root_last);
}

int main()
{
    Tensor a = tensor(1, 2, {1, 2}, true), b = tensor(1, 2, {3, 4}, true), c = tensor(1, 2, {5, 6}, true);
    report("sum(a + b)", sum(add(a, b)));
    Tensor s = add(a, b);
    report("a reached twice", sum(add(s, a)));
    report("wider graph", sum(add(mul(add(a, b), add(b, c)), a)));
    report("with a constant", sum(add(a, tensor(1, 2, {1, 1}))));
    report("a single leaf", a);
    return 0;
}
"""


def topological_order_program(source):
    return checked_program([("your topological_order", source)], TOPOLOGY_MAIN)


topological_order_cpp = practice_native("topological_order", topological_order_cpp,
                                        topological_order_cpp_reference, topological_order_program, language="c++")

**Expected behaviour.** Each line reads *count* `1 1 1`: 4 nodes for `sum(a + b)`, 5 when `a` is reached twice (it must still appear once), 8 for the wider graph, 4 with the constant and 1 for a single leaf. If your order repeats `a`, the third number turns to 0. If you forget to return early for a seen node, the recursion visits shared subgraphs again and again.

## 3 · The backward pass

`backward(root)` runs the rules in reverse topological order:

1. **Start afresh in the middle.** Every node that has a rule is an intermediate result; set its gradient to zeros, so that a second `backward` on the same graph does not add stale intermediate gradients. Leaves, such as the weights, are left alone: their gradients **accumulate** across calls, as in PyTorch, which is why training clears them with `zero_grad` before each step.
2. **Seed the root.** $\partial L / \partial L = 1$, so fill the root's gradient with ones (`grad_of(*root)` gives the buffer).
3. **Walk backwards.** Visit the order from the end to the start and call the rule of every node that has one, passing the node itself: `node->backward_rule(*node)`.

`order.rbegin()` and `order.rend()` iterate a vector backwards, and `*it` is the element. An empty `std::function` converts to `false`, so `if (node->backward_rule)` tests whether a node has a rule.

### ✍️ Your turn: `backward`

Complete `backward(root)` with the three steps above. Use your `topological_order`, which the check compiles in.

*Type it rather than pasting it: typing is how the order sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def backward_cpp_reference():
    return r"""
void backward(const Tensor& root)
{
    std::vector<Node*> order = topological_order(root);
    for (Node* node : order)
        if (node->backward_rule)
            node->grad.assign(node->size(), 0.0);  // intermediate gradients start afresh
    std::vector<double>& seed = grad_of(*root);
    std::fill(seed.begin(), seed.end(), 1.0);
    for (auto it = order.rbegin(); it != order.rend(); ++it)
        if ((*it)->backward_rule)
            (*it)->backward_rule(**it);
}
"""

In [ ]:
def backward_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
void backward(const Tensor& root)
{
    std::vector<Node*> order = topological_order(root);
    /* ✍️ Your code here: reset the intermediate gradients, seed the root with ones, run the rules backwards */
}
"""

In [ ]:
BACKWARD_MAIN = r"""
int main()
{
    Tensor a = tensor(1, 3, {1, 2, 3}, true), b = tensor(1, 3, {4, 5, 6}, true);
    backward(sum(add(a, b)));
    show("sum(a + b): grad a", a->grad);
    show("sum(a + b): grad b", b->grad);

    Tensor p = tensor(1, 3, {1, 2, 3}, true), q = tensor(1, 3, {0.5, -1, 2}, true);
    backward(sum(mul(add(p, p), q)));                  // p is used twice
    show("p used twice: grad p", p->grad);
    show("p used twice: grad q", q->grad);

    Tensor x = tensor(2, 3, numbers(6, 0.1), true), bias = tensor(1, 3, {0, 0, 0}, true);
    Tensor w = tensor(2, 3, numbers(6, 1.7));          // fixed weights make the upstream gradient irregular
    backward(sum(mul(add_bias(x, bias), w)));
    show("bias: grad x", x->grad);
    show("bias: grad bias", bias->grad);

    Tensor v = tensor(1, 2, {1, -1}, true);
    Tensor loss = sum(mul(v, v));
    backward(loss);
    backward(loss);                                    // the leaf accumulates; the middle starts afresh
    show("two backward calls: grad v", v->grad);

    Tensor u = tensor(1, 2, {1, -1}, true), k = tensor(1, 2, {2, 2});
    backward(sum(mul(u, k)));
    show("constant factor: grad u", u->grad);
    std::printf("the constant has a gradient: %d\n", !k->grad.empty());
    return 0;
}
"""


def backward_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("your backward", source)],
                           BACKWARD_MAIN)


backward_cpp = practice_native("backward", backward_cpp, backward_cpp_reference, backward_program, language="c++")

**Expected behaviour.** `sum(a + b)` gives ones for both. With `p` used twice, `p` collects $2q$ = `[1, -2, 4]` and `q` gets $2p$ = `[2, 4, 6]`. The bias collects the column sums of `w`. After two backward calls on `sum(v * v)`, `v` holds `[4, -4]`, twice $2v$: the leaf accumulated, but the intermediate `mul` node started afresh each time. If you skip step 1, the second call adds the first call's intermediate gradient again and you get `[6, -6]`.

## 4 · Matrix multiplication, forward and backward

`matmul(a, b)` follows the pattern of the operations in the core. Check the shapes, create the output with `result`, compute $C = AB$ with the loop from notebook 1, and attach a rule. The rule implements the gradients you derived there, but **adds** them into the parents' gradients, because a parent used in several places collects a contribution from each:

$$\bar A \mathrel{+}= \bar C B^\top, \qquad \bar B \mathrel{+}= A^\top \bar C.$$

Inside the rule, `self` is the output node: `self.grad` is $\bar C$, `self.parents[0]` and `self.parents[1]` are $A$ and $B$, and `grad_of(a)` gives $A$'s gradient buffer. The lambda captures the three sizes by value, `[n, k, m]`, so it carries its own copy of them. Update a parent's gradient only when that parent `requires_grad`: a constant has no gradient buffer to fill.

### ✍️ Your turn: `matmul`

Complete `matmul(a, b)`:

- **inputs:** `a` ($n \times k$) and `b` ($k \times m$);
- **output:** a new tensor $C = AB$ ($n \times m$) whose rule adds $\bar C B^\top$ into `a`'s gradient and $A^\top \bar C$ into `b`'s, each only when that parent requires a gradient;
- **errors:** throw `std::invalid_argument` when `a->cols != b->rows`. The first lines are written for you.

Hints: the forward loop is notebook 1's `matmul` with `a->data[i * k + r]` and `b->data[r * m + j]`. In the rule, take `Node& a = *self.parents[0];`, then write two loop nests like notebook 1's `matmul_backward`, with `+=` instead of `=`.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_cpp_reference():
    return r"""
Tensor matmul(const Tensor& a, const Tensor& b)
{
    if (a->cols != b->rows)
        throw std::invalid_argument("matmul: the inner sizes differ");
    int n = a->rows, k = a->cols, m = b->cols;
    Tensor out = result(n, m, {a, b}, "matmul");
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < m; ++j) {
            double s = 0.0;
            for (int r = 0; r < k; ++r)
                s += a->data[i * k + r] * b->data[r * m + j];
            out->data[i * m + j] = s;
        }
    if (out->requires_grad)
        out->backward_rule = [n, k, m](Node& self) {
            Node& a = *self.parents[0];
            Node& b = *self.parents[1];
            if (a.requires_grad) {
                std::vector<double>& ga = grad_of(a);
                for (int i = 0; i < n; ++i)
                    for (int r = 0; r < k; ++r) {
                        double s = 0.0;
                        for (int j = 0; j < m; ++j)
                            s += self.grad[i * m + j] * b.data[r * m + j];
                        ga[i * k + r] += s;
                    }
            }
            if (b.requires_grad) {
                std::vector<double>& gb = grad_of(b);
                for (int r = 0; r < k; ++r)
                    for (int j = 0; j < m; ++j) {
                        double s = 0.0;
                        for (int i = 0; i < n; ++i)
                            s += a.data[i * k + r] * self.grad[i * m + j];
                        gb[r * m + j] += s;
                    }
            }
        };
    return out;
}
"""

In [ ]:
def matmul_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
Tensor matmul(const Tensor& a, const Tensor& b)
{
    if (a->cols != b->rows)
        throw std::invalid_argument("matmul: the inner sizes differ");
    int n = a->rows, k = a->cols, m = b->cols;
    Tensor out = result(n, m, {a, b}, "matmul");
    /* ✍️ Your code here: the forward loop */

    if (out->requires_grad)
        out->backward_rule = [n, k, m](Node& self) {
            /* ✍️ Your code here: add dC B^T into a's gradient and A^T dC into b's, when each requires one */
        };
    return out;
}
"""

In [ ]:
MATMUL_MAIN = r"""
int main()
{
    Tensor a = tensor(2, 3, {1, 2, 3, 4, 5, 6}, true), b = tensor(3, 2, {7, 8, 9, 10, 11, 12}, true);
    Tensor c = matmul(a, b);
    show("forward, 2x3 times 3x2", c->data);
    backward(sum(mul(c, tensor(2, 2, {1, -1, 0.5, 2}))));
    show("grad a", a->grad);
    show("grad b", b->grad);

    Tensor p = tensor(4, 5, numbers(20, 0.3), true), q = tensor(5, 3, numbers(15, 1.1), true);
    backward(sum(mul(matmul(p, q), tensor(4, 3, numbers(12, 2.5)))));
    show("grad p, 4x5 times 5x3", p->grad);
    show("grad q, 4x5 times 5x3", q->grad);

    Tensor x = tensor(2, 3, numbers(6, 0.2), true), fixed = tensor(3, 2, numbers(6, 0.9));
    backward(sum(matmul(x, fixed)));
    show("constant right-hand side: grad x", x->grad);
    std::printf("the constant got a gradient: %d\n", !fixed->grad.empty());

    try {
        matmul(a, a);
        std::printf("2x3 times 2x3: no error\n");
    } catch (const std::invalid_argument&) {
        std::printf("2x3 times 2x3: invalid_argument\n");
    }
    return 0;
}
"""


def matmul_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("your matmul", source)], MATMUL_MAIN)


matmul_cpp = practice_native("matmul", matmul_cpp, matmul_cpp_reference, matmul_program, language="c++")

**Expected behaviour.** The forward product is `58 64 139 154`, as in notebook 1, and the gradients match the reference. With a constant right-hand side only `x` receives a gradient, `fixed` stays without one, and multiplying a 2 × 3 matrix by another 2 × 3 matrix throws. If your gradients are twice too large or the constant case crashes, check the `requires_grad` tests in the rule.

### Check the rule against central differences

The check above compared your rule with the reference rule. The cell below compares it with numbers alone: it perturbs every entry of `a` and `b` in a small product, measures the change in a scalar loss, and reports the relative error for each, as in notebook 1. A value around $10^{-10}$ means the rule is right.

In [ ]:
GRADIENT_CHECK_MAIN = r"""
static double relative_error(const std::function<Tensor()>& loss_of, const Tensor& t)
{
    t->grad.clear();
    backward(loss_of());
    std::vector<double> analytic = t->grad;
    double diff = 0.0, scale_num = 0.0, scale_an = 0.0;
    for (size_t i = 0; i < t->size(); ++i) {
        double saved = t->data[i];
        t->data[i] = saved + 1e-6;
        double plus = loss_of()->data[0];
        t->data[i] = saved - 1e-6;
        double minus = loss_of()->data[0];
        t->data[i] = saved;
        double numeric = (plus - minus) / 2e-6;
        diff += (numeric - analytic[i]) * (numeric - analytic[i]);
        scale_num += numeric * numeric;
        scale_an += analytic[i] * analytic[i];
    }
    return std::sqrt(diff) / std::max(std::sqrt(scale_num) + std::sqrt(scale_an), 1e-12);
}

int main()
{
    Tensor a = tensor(3, 4, numbers(12, 0.4), true), b = tensor(4, 2, numbers(8, 1.9), true);
    Tensor w = tensor(3, 2, numbers(6, 3.1));
    auto loss_of = [&] { return sum(mul(matmul(a, b), w)); };
    std::printf("relative error for a: %.1e\nrelative error for b: %.1e\n",
                relative_error(loss_of, a), relative_error(loss_of, b));
    return 0;
}
"""
print(run(build(checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                                 ("matmul", matmul_cpp())], GRADIENT_CHECK_MAIN), "c++")))

## 5 · A fused, stable cross-entropy

The loss could be built from smaller operations, a log-softmax node and a selection node, but **fusing** them into one operation is both cheaper and safer. The forward pass computes the stable log-softmax of each row, exactly as in notebook 1, and the mean negative log-probability of the true classes. It also keeps the probabilities $p_{ij}$, because the backward rule needs them:

$$\frac{\partial L}{\partial z_{ij}} = \frac{p_{ij} - [j = y_i]}{n}.$$

The output is a $1 \times 1$ tensor. Its rule must multiply by the upstream gradient `self.grad[0]`, which is 1 when the loss is the root but not when the loss is scaled or added to another term.

The rule needs the probabilities after `cross_entropy` has returned, so the lambda must **own** a copy of them. `[probs = std::move(probs), labels, n, c]` is an *init-capture*: it moves the vector into the lambda instead of copying it, and captures the labels and sizes by value. Never capture a local variable by reference (`[&]`) in a rule: the rule runs later, after the local is gone, and would read freed memory.

### ✍️ Your turn: `cross_entropy`

Complete `cross_entropy(logits, labels)`:

- **inputs:** `logits`, an $n \times c$ tensor; `labels`, one class index per row;
- **output:** a $1 \times 1$ tensor holding $-\frac1n \sum_i \log \operatorname{softmax}(z_i)_{y_i}$, computed with each row's maximum subtracted, whose rule adds `self.grad[0] / n * (p - onehot)` into the logits' gradient;
- **errors:** throw `std::invalid_argument` when the number of labels differs from the number of rows (written for you).

Hints: for row `i`, `const double* z = &logits->data[i * c];` points at its first logit and `*std::max_element(z, z + c)` is its maximum. The row's log-probability of class `j` is `z[j] - largest - std::log(sum)`, and its probability is `std::exp(z[j] - largest) / sum`.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def cross_entropy_cpp_reference():
    return r"""
Tensor cross_entropy(const Tensor& logits, const std::vector<int>& labels)
{
    int n = logits->rows, c = logits->cols;
    if (static_cast<int>(labels.size()) != n)
        throw std::invalid_argument("cross_entropy: one label per row");
    std::vector<double> probs(logits->size());
    double total = 0.0;
    for (int i = 0; i < n; ++i) {
        const double* z = &logits->data[i * c];
        double largest = *std::max_element(z, z + c);
        double sum = 0.0;
        for (int j = 0; j < c; ++j)
            sum += std::exp(z[j] - largest);
        for (int j = 0; j < c; ++j)
            probs[i * c + j] = std::exp(z[j] - largest) / sum;
        total -= z[labels[i]] - largest - std::log(sum);
    }
    Tensor out = result(1, 1, {logits}, "cross_entropy");
    out->data[0] = total / n;
    if (out->requires_grad)
        out->backward_rule = [probs = std::move(probs), labels, n, c](Node& self) {
            std::vector<double>& g = grad_of(*self.parents[0]);
            double upstream = self.grad[0] / n;
            for (int i = 0; i < n; ++i)
                for (int j = 0; j < c; ++j)
                    g[i * c + j] += upstream * (probs[i * c + j] - (j == labels[i] ? 1.0 : 0.0));
        };
    return out;
}
"""

In [ ]:
def cross_entropy_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
Tensor cross_entropy(const Tensor& logits, const std::vector<int>& labels)
{
    int n = logits->rows, c = logits->cols;
    if (static_cast<int>(labels.size()) != n)
        throw std::invalid_argument("cross_entropy: one label per row");
    std::vector<double> probs(logits->size());
    double total = 0.0;
    /* ✍️ Your code here: for each row, the maximum, the sum of exponentials, the probabilities
          and the row's negative log-probability of its label, added to total */

    Tensor out = result(1, 1, {logits}, "cross_entropy");
    out->data[0] = total / n;
    if (out->requires_grad)
        out->backward_rule = [probs = std::move(probs), labels, n, c](Node& self) {
            /* ✍️ Your code here: add self.grad[0] / n * (p - onehot) into the logits' gradient */
        };
    return out;
}
"""

In [ ]:
CROSS_ENTROPY_MAIN = r"""
int main()
{
    Tensor z = tensor(3, 2, {std::log(0.25), std::log(0.75), 0.0, 0.0, 2.0, -1.0}, true);
    Tensor loss = cross_entropy(z, {1, 0, 1});
    show("loss, 3 rows", loss->data);
    backward(loss);
    show("grad, 3 rows", z->grad);

    Tensor big = tensor(1, 3, {1000, 1001, 1002}, true);
    Tensor loss_big = cross_entropy(big, {0});
    backward(loss_big);
    show("loss, logits near 1000", loss_big->data);
    show("grad, logits near 1000", big->grad);

    Tensor r = tensor(4, 5, numbers(20, 0.7), true);
    Tensor scaled = mul(cross_entropy(r, {4, 0, 2, 2}), tensor(1, 1, {3.0}));
    backward(scaled);                                  // the rule must use its upstream gradient, 3
    show("3 x loss", scaled->data);
    show("grad of 3 x loss", r->grad);

    try {
        cross_entropy(r, {0, 1});
        std::printf("two labels for four rows: no error\n");
    } catch (const std::invalid_argument&) {
        std::printf("two labels for four rows: invalid_argument\n");
    }
    return 0;
}
"""


def cross_entropy_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("matmul", matmul_cpp()), ("your cross_entropy", source)], CROSS_ENTROPY_MAIN)


cross_entropy_cpp = practice_native("cross_entropy", cross_entropy_cpp, cross_entropy_cpp_reference,
                                    cross_entropy_program, language="c++")

**Expected behaviour.** The first row, with probabilities 0.25 and 0.75 and label 1, contributes $-\log 0.75 \approx 0.288$, the second, with equal logits, $\log 2 \approx 0.693$, and the third, whose label has the smaller logit, about 3.049, so the mean loss is about 1.3431. Logits near 1000 give a finite loss of about 2.4076, the same as `[1, 2, 3]` would. When the loss is multiplied by 3, its gradient is three times larger: if yours is not, the rule ignored `self.grad[0]`.

### 🧪 Try it: a cycle that never frees

A rule must not capture its own output node. The program below adds `relu_leaky`, a copy of `relu` whose rule captures `out`: the node owns its rule, and the rule owns the node, so the count of owners can never reach zero. Run it and compare the live-node counts after each graph goes out of scope.

In [ ]:
LEAK_DEMO = r"""
Tensor relu_leaky(const Tensor& x)
{
    Tensor out = result(x->rows, x->cols, {x}, "relu_leaky");
    for (size_t i = 0; i < x->size(); ++i)
        out->data[i] = x->data[i] > 0.0 ? x->data[i] : 0.0;
    if (out->requires_grad)
        out->backward_rule = [out](Node&) {            // the bug: the rule owns its own node
            Node& x = *out->parents[0];
            std::vector<double>& g = grad_of(x);
            for (size_t i = 0; i < g.size(); ++i)
                if (x.data[i] > 0.0)
                    g[i] += out->grad[i];
        };
    return out;
}

int main()
{
    {
        Tensor x = tensor(2, 2, {1, -1, 2, -2}, true);
        Tensor loss = sum(relu(x));
        backward(loss);
        std::printf("while the correct graph exists: %ld live nodes\n", live_nodes);
    }
    std::printf("after it goes out of scope:     %ld live nodes\n", live_nodes);
    {
        Tensor x = tensor(2, 2, {1, -1, 2, -2}, true);
        Tensor loss = sum(relu_leaky(x));
        backward(loss);
    }
    std::printf("after the leaky graph's scope:  %ld live nodes\n", live_nodes);
    return 0;
}
"""
print(run(build(engine(("topological_order", topological_order_cpp()), ("backward", backward_cpp())) + LEAK_DEMO,
                "c++")))

**What to notice.** The correct graph holds three nodes and frees all of them. The leaky one leaves **two** nodes behind for ever: the `relu_leaky` node, kept alive by its own rule, and `x`, kept alive as its parent. A training loop builds a new graph every step, so such a leak grows by a few nodes per step until the process runs out of memory. Passing the node to its rule as an argument, as the engine does, avoids the cycle. (`std::weak_ptr` is the other standard way out of a cycle: a pointer that observes without owning.)

## 6 · Modules, a generator you can save, and Adam

The cell below adds the pieces around the engine:

- **`Rng`**, the *splitmix64* generator. Its whole state is a single 64-bit integer, so a checkpoint can store it in eight bytes and training can resume the same random sequence. `normal()` turns two uniform numbers into a Gaussian one with the Box-Muller method.
- **`Linear`** and **`Mlp`**, the modules: a `Linear` owns its weight `W` and bias `b`, initialised with He's scale $\sqrt{2 / \text{inputs}}$, and calling it on `x` builds `add_bias(matmul(x, W), b)` in the graph. `Mlp` is the 2 → hidden → classes network.
- **`zero_grad`**, and **`Adam`**, the optimiser's state: one first-moment and one second-moment vector per parameter, and the step counter. The `step` method is declared in the `struct` and defined later, by you.

Adam keeps running averages of each gradient $g$ and of its square, corrects their bias towards zero at the start, and scales each parameter's step by them:

$$m \leftarrow \beta_1 m + (1 - \beta_1) g, \qquad v \leftarrow \beta_2 v + (1 - \beta_2) g^2,$$

$$\theta \leftarrow \theta - \eta \, \frac{m / (1 - \beta_1^t)}{\sqrt{v / (1 - \beta_2^t)} + \varepsilon},$$

where $t$ is the step number, counted from 1. On the first step $m / (1 - \beta_1) = g$ and $v / (1 - \beta_2) = g^2$, so every parameter with a nonzero gradient moves by almost exactly the learning rate $\eta$.

In [ ]:
MODULES_CPP = r"""
// splitmix64: a small, fast generator whose whole state is one 64-bit number.
struct Rng {
    uint64_t state;
    explicit Rng(uint64_t seed) : state(seed) {}
    uint64_t next()
    {
        uint64_t z = (state += 0x9E3779B97F4A7C15ull);
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ull;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBull;
        return z ^ (z >> 31);
    }
    double uniform() { return static_cast<double>(next() >> 11) * 0x1.0p-53; }   // in [0, 1)
    double normal()                                                              // Box-Muller
    {
        double u = uniform(), v = uniform();
        return std::sqrt(-2.0 * std::log(1.0 - u)) * std::cos(6.283185307179586 * v);
    }
};

struct Linear {
    Tensor W, b;
    Linear(int inputs, int outputs, Rng& rng)
        : W(std::make_shared<Node>(inputs, outputs, true, "W")), b(std::make_shared<Node>(1, outputs, true, "b"))
    {
        double scale = std::sqrt(2.0 / inputs);                              // He initialisation
        for (double& w : W->data)
            w = scale * rng.normal();
    }
    Tensor operator()(const Tensor& x) const { return add_bias(matmul(x, W), b); }
};

struct Mlp {
    Linear first, second;
    Mlp(int inputs, int hidden, int classes, Rng& rng) : first(inputs, hidden, rng), second(hidden, classes, rng) {}
    Tensor operator()(const Tensor& x) const { return second(relu(first(x))); }
    std::vector<Tensor> parameters() const { return {first.W, first.b, second.W, second.b}; }
};

void zero_grad(const std::vector<Tensor>& params)
{
    for (const Tensor& p : params)
        p->grad.assign(p->size(), 0.0);
}

struct Adam {
    std::vector<Tensor> params;
    double learning_rate, beta1 = 0.9, beta2 = 0.999, eps = 1e-8;
    long step_count = 0;
    std::vector<std::vector<double>> m, v;   // one first and one second moment per parameter entry
    Adam(std::vector<Tensor> parameters, double lr) : params(std::move(parameters)), learning_rate(lr)
    {
        for (const Tensor& p : params) {
            m.emplace_back(p->size(), 0.0);
            v.emplace_back(p->size(), 0.0);
        }
    }
    void step();                             // defined below, outside the struct
};
"""

### ✍️ Your turn: `Adam::step`

Complete `Adam::step()`, a member function defined outside its `struct`: inside it, `params`, `m`, `v`, `beta1`, `step_count` and the other members are in scope by name.

- add 1 to `step_count`, then compute the two corrections $1 - \beta_1^t$ and $1 - \beta_2^t$ with `std::pow`;
- for each parameter `p` (an index into `params`, `m` and `v`), skip it when its gradient is empty;
- otherwise, for each entry `i`, update `m[p][i]` and `v[p][i]`, then the parameter's `data[i]`, with the formulas above.

*Type it rather than pasting it: typing is how the formulas sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def adam_step_cpp_reference():
    return r"""
void Adam::step()
{
    ++step_count;
    double correction1 = 1.0 - std::pow(beta1, static_cast<double>(step_count));
    double correction2 = 1.0 - std::pow(beta2, static_cast<double>(step_count));
    for (size_t p = 0; p < params.size(); ++p) {
        Node& param = *params[p];
        if (param.grad.empty())
            continue;
        for (size_t i = 0; i < param.size(); ++i) {
            double g = param.grad[i];
            m[p][i] = beta1 * m[p][i] + (1.0 - beta1) * g;
            v[p][i] = beta2 * v[p][i] + (1.0 - beta2) * g * g;
            param.data[i] -= learning_rate * (m[p][i] / correction1) / (std::sqrt(v[p][i] / correction2) + eps);
        }
    }
}
"""

In [ ]:
def adam_step_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
void Adam::step()
{
    /* ✍️ Your code here: the step counter, the corrections, then the loops over parameters and entries */
}
"""

In [ ]:
ADAM_MAIN = r"""
int main()
{
    Tensor w = tensor(1, 3, {1.0, -2.0, 0.5}, true), unused = tensor(1, 2, {7.0, 8.0}, true);
    Adam adam({w, unused}, 0.1);
    for (int step = 1; step <= 3; ++step) {
        w->grad = {0.5 * step, -1.0, 0.0};             // unused keeps an empty gradient: skip it
        adam.step();
        char label[32];
        std::snprintf(label, sizeof label, "after step %d", step);
        show(label, w->data);
    }
    show("first moments", adam.m[0]);
    show("second moments", adam.v[0]);
    show("parameter without a gradient", unused->data);
    std::printf("step count: %ld\n", adam.step_count);
    return 0;
}
"""


def adam_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("matmul", matmul_cpp()), ("cross_entropy", cross_entropy_cpp()),
                            ("modules", MODULES_CPP), ("your Adam::step", source)], ADAM_MAIN)


adam_step_cpp = practice_native("Adam::step", adam_step_cpp, adam_step_cpp_reference, adam_program, language="c++")

**Expected behaviour.** After the first step the weights are about `[0.9, -1.9, 0.5]`: each entry with a nonzero gradient moved by the learning rate, 0.1, against its gradient's sign, and the entry with a zero gradient did not move. The parameter with an empty gradient is untouched, and the counter reads 3.

## 7 · One correct training step

A training step is a fixed sequence of state changes:

1. clear the parameters' gradients, because `backward` adds to them;
2. run the model and the loss, which records the graph;
3. stop with an error if the loss is not finite, before a `nan` reaches the weights;
4. run `backward` on the loss;
5. let the optimiser update the weights.

The graph of the step is destroyed when the function returns: the `loss` variable is its last owner. Nothing in C++ needs to be freed by hand.

### ✍️ Your turn: `train_step`

Complete `train_step(model, adam, x, labels)`, returning the loss as a `double`:

- `zero_grad(model.parameters())`;
- `Tensor loss = cross_entropy(model(x), labels);`
- throw `std::runtime_error` with a helpful message when `!std::isfinite(loss->data[0])`;
- `backward(loss);`, then `adam.step();`, then return `loss->data[0]`.

*Type it rather than pasting it: typing is how the order sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def train_step_cpp_reference():
    return r"""
double train_step(Mlp& model, Adam& adam, const Tensor& x, const std::vector<int>& labels)
{
    zero_grad(model.parameters());
    Tensor loss = cross_entropy(model(x), labels);
    if (!std::isfinite(loss->data[0]))
        throw std::runtime_error("the loss is not finite: check the learning rate and the data");
    backward(loss);
    adam.step();
    return loss->data[0];
}
"""

In [ ]:
def train_step_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
double train_step(Mlp& model, Adam& adam, const Tensor& x, const std::vector<int>& labels)
{
    /* ✍️ Your code here (about six lines) */
}
"""

In [ ]:
TRAIN_STEP_MAIN = r"""
int main()
{
    Rng rng(5);
    Mlp model(2, 4, 3, rng);
    Adam adam(model.parameters(), 0.05);
    Tensor x = tensor(6, 2, numbers(12, 0.4));
    std::vector<int> y = {0, 1, 2, 2, 1, 0};
    std::vector<double> losses;
    for (int s = 0; s < 4; ++s)
        losses.push_back(train_step(model, adam, x, y));
    show("losses of 4 steps", losses);
    show("W2 after 4 steps", model.second.W->data);
    show("b1 after 4 steps", model.first.b->data);
    std::printf("Adam steps: %ld, live nodes: %ld\n", adam.step_count, live_nodes);
    return 0;
}
"""


def train_step_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("matmul", matmul_cpp()), ("cross_entropy", cross_entropy_cpp()),
                            ("modules", MODULES_CPP), ("Adam::step", adam_step_cpp()),
                            ("your train_step", source)], TRAIN_STEP_MAIN)


train_step_cpp = practice_native("train_step", train_step_cpp, train_step_cpp_reference, train_step_program,
                                 language="c++")

**Expected behaviour.** The four losses fall, the weights match the reference, and only 5 nodes are alive afterwards: the four parameters and `x`. Every graph built during the steps was freed. If you forget `zero_grad`, each step adds the new gradients to all the old ones, the weights drift away from the reference's, and the check shows the first loss that differs.

## 8 · Checkpoints that resume exactly

A checkpoint for **resuming** training must hold everything the next update depends on: the weights, Adam's moments and step counter, and the random generator's state, which decides the next mini-batch. With all of it, a run that stops and restarts produces exactly the same numbers as a run that never stopped.

The `Trainer` below bundles that state, and `sample_batch` draws a mini-batch with replacement from its generator. The format is binary, so each `double` is stored exactly:

| bytes | content |
|---|---|
| 8 | the **magic** `FURINAC1`, which identifies the file type |
| 4 | the format **version**, 1 |
| 3 × 4 | the sizes: inputs, hidden units, classes |
| 8 + 8 + 8 | the learning rate, the generator's state and Adam's step counter |
| per parameter | its values, then Adam's first moments, then its second moments, in the order W1, b1, W2, b2 |

The numbers are written in the machine's own byte order, which is little-endian on every x86-64 and ARM computer you are likely to use; a format meant for other machines would fix the byte order explicitly. A loader must check the magic and the version before trusting anything else, and must notice a file that ends too early.

Files are a resource like memory, so the code holds them in a `std::unique_ptr` with `fclose` as its *deleter*: the file is closed when the pointer goes away, even when an exception leaves the function early. `read_exact` throws when the file ends before the requested bytes.

In [ ]:
TRAINER_CPP = r"""
// Everything that training changes, in one place: the generator, the model and the optimiser.
// Members are initialised in the order they are declared, so rng exists before model uses it.
struct Trainer {
    int inputs, hidden, classes;
    Rng rng;
    Mlp model;
    Adam adam;
    Trainer(int inputs_, int hidden_, int classes_, double learning_rate, uint64_t seed)
        : inputs(inputs_), hidden(hidden_), classes(classes_), rng(seed),
          model(inputs_, hidden_, classes_, rng), adam(model.parameters(), learning_rate) {}
};

// A mini-batch of `batch` rows drawn with replacement, using the trainer's generator.
Tensor sample_batch(Trainer& t, const double* X, const int* y, int rows, int batch, std::vector<int>& labels)
{
    Tensor x = std::make_shared<Node>(batch, t.inputs, false, "batch");
    labels.assign(batch, 0);
    for (int i = 0; i < batch; ++i) {
        int row = static_cast<int>(t.rng.next() % static_cast<uint64_t>(rows));
        std::copy(X + static_cast<size_t>(row) * t.inputs, X + static_cast<size_t>(row + 1) * t.inputs,
                  x->data.begin() + static_cast<size_t>(i) * t.inputs);
        labels[i] = y[row];
    }
    return x;
}
"""

In [ ]:
CHECKPOINT_CPP = r"""
// File format, little-endian: "FURINAC1", uint32 version, int32 inputs, hidden, classes,
// double learning rate, uint64 generator state, int64 Adam step, then for each parameter
// in order (W1, b1, W2, b2): its values, Adam's first moments and Adam's second moments.
static const char MAGIC[8] = {'F', 'U', 'R', 'I', 'N', 'A', 'C', '1'};
static const uint32_t VERSION = 1;

using File = std::unique_ptr<std::FILE, int (*)(std::FILE*)>;   // fclose runs when a File goes away

File open_file(const char* path, const char* mode)
{
    File file(std::fopen(path, mode), &std::fclose);
    if (!file)
        throw std::runtime_error(std::string("cannot open ") + path);
    return file;
}

void write_exact(std::FILE* f, const void* data, size_t bytes)
{
    if (std::fwrite(data, 1, bytes, f) != bytes)
        throw std::runtime_error("could not write the whole checkpoint");
}

void read_exact(std::FILE* f, void* data, size_t bytes)
{
    if (std::fread(data, 1, bytes, f) != bytes)
        throw std::runtime_error("the checkpoint is truncated");
}

void save_checkpoint(const Trainer& t, const char* path)
{
    File file = open_file(path, "wb");
    int32_t sizes[3] = {t.inputs, t.hidden, t.classes};
    int64_t steps = t.adam.step_count;
    write_exact(file.get(), MAGIC, sizeof MAGIC);
    write_exact(file.get(), &VERSION, sizeof VERSION);
    write_exact(file.get(), sizes, sizeof sizes);
    write_exact(file.get(), &t.adam.learning_rate, sizeof(double));
    write_exact(file.get(), &t.rng.state, sizeof(uint64_t));
    write_exact(file.get(), &steps, sizeof steps);
    for (size_t p = 0; p < t.adam.params.size(); ++p) {
        size_t bytes = t.adam.params[p]->size() * sizeof(double);
        write_exact(file.get(), t.adam.params[p]->data.data(), bytes);
        write_exact(file.get(), t.adam.m[p].data(), bytes);
        write_exact(file.get(), t.adam.v[p].data(), bytes);
    }
}
"""

### ✍️ Your turn: `load_checkpoint`

Complete `load_checkpoint(path)`, the mirror image of `save_checkpoint`:

- open the file for reading (`open_file(path, "rb")`) and read the magic with `read_exact`; if `std::memcmp(magic, MAGIC, sizeof magic) != 0`, throw `std::runtime_error("not a Furina checkpoint")`;
- read the version and throw if it is not `VERSION`;
- read the three sizes, the learning rate, the generator's state and the step counter, in that order;
- create the trainer with `std::make_unique<Trainer>(sizes[0], sizes[1], sizes[2], learning_rate, 0)`, then overwrite its `rng.state` and `adam.step_count`;
- for each parameter `p`, read its values, `adam.m[p]` and `adam.v[p]`, each `params[p]->size() * sizeof(double)` bytes, into the vectors' memory (`.data()`);
- return the trainer.

The seed 0 passed to the constructor does not matter: every number it produces is overwritten by the file.

*Type it rather than pasting it: typing is how the format sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def load_checkpoint_cpp_reference():
    return r"""
std::unique_ptr<Trainer> load_checkpoint(const char* path)
{
    File file = open_file(path, "rb");
    char magic[8];
    uint32_t version;
    int32_t sizes[3];
    double learning_rate;
    uint64_t state;
    int64_t steps;
    read_exact(file.get(), magic, sizeof magic);
    if (std::memcmp(magic, MAGIC, sizeof magic) != 0)
        throw std::runtime_error("not a Furina checkpoint");
    read_exact(file.get(), &version, sizeof version);
    if (version != VERSION)
        throw std::runtime_error("unsupported checkpoint version " + std::to_string(version));
    read_exact(file.get(), sizes, sizeof sizes);
    read_exact(file.get(), &learning_rate, sizeof learning_rate);
    read_exact(file.get(), &state, sizeof state);
    read_exact(file.get(), &steps, sizeof steps);
    auto t = std::make_unique<Trainer>(sizes[0], sizes[1], sizes[2], learning_rate, 0);
    t->rng.state = state;
    t->adam.step_count = steps;
    for (size_t p = 0; p < t->adam.params.size(); ++p) {
        size_t bytes = t->adam.params[p]->size() * sizeof(double);
        read_exact(file.get(), t->adam.params[p]->data.data(), bytes);
        read_exact(file.get(), t->adam.m[p].data(), bytes);
        read_exact(file.get(), t->adam.v[p].data(), bytes);
    }
    return t;
}
"""

In [ ]:
def load_checkpoint_cpp():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
std::unique_ptr<Trainer> load_checkpoint(const char* path)
{
    File file = open_file(path, "rb");
    char magic[8];
    uint32_t version;
    int32_t sizes[3];
    double learning_rate;
    uint64_t state;
    int64_t steps;
    /* ✍️ Your code here: read and check the header, create the trainer, then read every parameter */
}
"""

In [ ]:
def c_string(text):
    return '"' + text.replace("\\", "\\\\").replace('"', '\\"') + '"'


CHECKPOINT_FILES = {name: os.path.join(BUILD_DIR, name) for name in ("check.ckpt", "damaged.ckpt", "missing.ckpt")}
LOAD_MAIN = r"""
static std::vector<char> read_all(const char* path)
{
    File f = open_file(path, "rb");
    std::vector<char> bytes;
    char buffer[4096];
    size_t got;
    while ((got = std::fread(buffer, 1, sizeof buffer, f.get())) > 0)
        bytes.insert(bytes.end(), buffer, buffer + got);
    return bytes;
}

static void write_all(const char* path, const std::vector<char>& bytes)
{
    File f = open_file(path, "wb");
    write_exact(f.get(), bytes.data(), bytes.size());
}

static void try_loading(const char* label, const char* path)
{
    try {
        load_checkpoint(path);
        std::printf("%s: accepted\n", label);
    } catch (const std::exception& error) {
        std::printf("%s: rejected\n", label);
        std::fprintf(stderr, "%s: %s\n", label, error.what());
    }
}

int main()
{
    const char* path = PATH_CHECK;
    std::vector<double> X = numbers(20, 0.6);
    std::vector<int> y = {0, 1, 2, 0, 1, 2, 0, 1, 2, 0}, labels;
    Trainer original(2, 4, 3, 0.05, 9);
    for (int s = 0; s < 5; ++s)
        train_step(original.model, original.adam, sample_batch(original, X.data(), y.data(), 10, 4, labels), labels);
    save_checkpoint(original, path);

    std::unique_ptr<Trainer> loaded = load_checkpoint(path);
    std::printf("sizes: %d %d %d\n", loaded->inputs, loaded->hidden, loaded->classes);
    std::printf("generator state and Adam steps: %llu %ld\n",
                static_cast<unsigned long long>(loaded->rng.state), loaded->adam.step_count);
    show("W1", loaded->model.first.W->data);
    show("b2", loaded->model.second.b->data);
    show("second moments of W2", loaded->adam.v[2]);

    std::vector<double> continued, resumed;
    for (int s = 0; s < 3; ++s) {
        continued.push_back(train_step(original.model, original.adam,
                                       sample_batch(original, X.data(), y.data(), 10, 4, labels), labels));
        resumed.push_back(train_step(loaded->model, loaded->adam,
                                     sample_batch(*loaded, X.data(), y.data(), 10, 4, labels), labels));
    }
    show("losses of the run that never stopped", continued);
    show("losses of the resumed run", resumed);

    std::vector<char> good = read_all(path), bad = good;
    bad[0] = 'X';                                      // a different file type
    write_all(PATH_DAMAGED, bad);
    try_loading("wrong magic", PATH_DAMAGED);
    bad = good;
    bad[8] = 99;                                       // the version is the four bytes after the magic
    write_all(PATH_DAMAGED, bad);
    try_loading("unknown version", PATH_DAMAGED);
    bad.assign(good.begin(), good.begin() + good.size() / 2);
    write_all(PATH_DAMAGED, bad);
    try_loading("truncated file", PATH_DAMAGED);
    try_loading("missing file", PATH_MISSING);
    return 0;
}
""".replace("PATH_CHECK", c_string(CHECKPOINT_FILES["check.ckpt"])) \
   .replace("PATH_DAMAGED", c_string(CHECKPOINT_FILES["damaged.ckpt"])) \
   .replace("PATH_MISSING", c_string(CHECKPOINT_FILES["missing.ckpt"]))


def load_checkpoint_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("matmul", matmul_cpp()), ("cross_entropy", cross_entropy_cpp()),
                            ("modules", MODULES_CPP), ("Adam::step", adam_step_cpp()), ("Trainer", TRAINER_CPP),
                            ("train_step", train_step_cpp()), ("checkpoints", CHECKPOINT_CPP),
                            ("your load_checkpoint", source)], LOAD_MAIN)


load_checkpoint_cpp = practice_native("load_checkpoint", load_checkpoint_cpp, load_checkpoint_cpp_reference,
                                      load_checkpoint_program, language="c++")

**Expected behaviour.** The loaded trainer has the original's sizes, generator state, step counter, weights and moments, and the resumed run's three losses equal the uninterrupted run's exactly. The four damaged files are all rejected. If you skip the magic or the version check, a damaged file is *accepted* and the check shows it; if the resumed losses differ, a piece of state was not restored.

## 9 · Put it together: drive the engine from Python

Python cannot call C++ functions or catch C++ exceptions directly, so the engine offers a small **C interface**. The functions are declared `extern "C"`, which gives them plain C names and calling conventions; a `Trainer` crosses the boundary as an opaque `void *`; and each function catches every exception, keeps its message for `last_error()` and reports failure with its return value. An exception that escaped into Python would end the whole process.

The cell compiles the engine, with every piece you wrote, into a shared library, declares the function types for `ctypes`, and trains a 2 → 32 → 3 network with Adam on mini-batches of 32 points from a three-armed spiral, a task no straight boundary can solve. Then it scores the training points and 900 fresh ones.

In [ ]:
C_API_CPP = r"""
// The functions Python calls. C has no exceptions or classes, so each function
// catches every exception, keeps its message for last_error(), and reports
// failure with a return value; a Trainer travels as an opaque void pointer.
static std::string error_message;

template <class Body>
static bool guarded(Body body)
{
    try {
        body();
        return true;
    } catch (const std::exception& error) {
        error_message = error.what();
        return false;
    }
}

extern "C" {

void* trainer_new(int inputs, int hidden, int classes, double learning_rate, uint64_t seed)
{
    Trainer* t = nullptr;
    guarded([&] { t = new Trainer(inputs, hidden, classes, learning_rate, seed); });
    return t;
}

void trainer_free(void* handle) { delete static_cast<Trainer*>(handle); }

double trainer_train(void* handle, const double* X, const int* y, int rows, int batch, int steps, double* losses)
{
    Trainer& t = *static_cast<Trainer*>(handle);
    double loss = NAN;
    bool ok = guarded([&] {
        std::vector<int> labels;
        for (int s = 0; s < steps; ++s) {
            Tensor x = sample_batch(t, X, y, rows, batch, labels);
            loss = train_step(t.model, t.adam, x, labels);
            if (losses)
                losses[s] = loss;
        }
    });
    return ok ? loss : NAN;
}

void trainer_logits(void* handle, const double* X, int rows, double* out)
{
    Trainer& t = *static_cast<Trainer*>(handle);
    Tensor x = std::make_shared<Node>(rows, t.inputs, false, "x");
    std::copy(X, X + static_cast<size_t>(rows) * t.inputs, x->data.begin());
    Tensor logits = t.model(x);
    std::copy(logits->data.begin(), logits->data.end(), out);
}

int trainer_save(void* handle, const char* path)
{
    return guarded([&] { save_checkpoint(*static_cast<Trainer*>(handle), path); }) ? 0 : -1;
}

void* trainer_load(const char* path)
{
    Trainer* t = nullptr;
    guarded([&] { t = load_checkpoint(path).release(); });
    return t;
}

uint64_t trainer_rng_state(void* handle) { return static_cast<Trainer*>(handle)->rng.state; }
long trainer_steps(void* handle) { return static_cast<Trainer*>(handle)->adam.step_count; }
const char* last_error() { return error_message.c_str(); }
long live_node_count() { return live_nodes; }

}
"""

In [ ]:
import time


def spirals(per_class, seed, classes=3, noise=0.2):
    """Three interleaved spiral arms in the plane; the label is the arm."""
    gen = random.Random(seed)
    X, y = array("d"), array("i")
    for c in range(classes):
        for i in range(per_class):
            r = i / per_class
            t = 4.0 * c + 4.0 * r + gen.gauss(0.0, noise)
            X.extend((r * math.sin(t), r * math.cos(t)))
            y.append(c)
    return X, y


def full_engine():
    return engine(("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                  ("matmul", matmul_cpp()), ("cross_entropy", cross_entropy_cpp()), ("modules", MODULES_CPP),
                  ("Adam::step", adam_step_cpp()), ("Trainer", TRAINER_CPP), ("train_step", train_step_cpp()),
                  ("checkpoints", CHECKPOINT_CPP), ("load_checkpoint", load_checkpoint_cpp()),
                  ("the C interface", C_API_CPP))


lib = load(full_engine(), "c++")
P, I, D = ctypes.c_void_p, ctypes.c_int, ctypes.c_double
for name, argtypes, restype in [
    ("trainer_new", [I, I, I, D, ctypes.c_uint64], P),
    ("trainer_free", [P], None),
    ("trainer_train", [P, P, P, I, I, I, P], D),
    ("trainer_logits", [P, P, I, P], None),
    ("trainer_save", [P, ctypes.c_char_p], I),
    ("trainer_load", [ctypes.c_char_p], P),
    ("trainer_rng_state", [P], ctypes.c_uint64),
    ("trainer_steps", [P], ctypes.c_long),
    ("last_error", [], ctypes.c_char_p),
    ("live_node_count", [], ctypes.c_long),
]:
    getattr(lib, name).argtypes, getattr(lib, name).restype = argtypes, restype


def accuracy(trainer, X, y, classes=3):
    logits = array("d", bytes(8 * classes * len(y)))
    lib.trainer_logits(trainer, address(X), len(y), address(logits))
    rows = [logits[i * classes:(i + 1) * classes] for i in range(len(y))]
    return sum(row.index(max(row)) == label for row, label in zip(rows, y)) / len(y)


X_train, y_train = spirals(100, seed=7)
X_fresh, y_fresh = spirals(300, seed=8)
trainer = lib.trainer_new(2, 32, 3, 0.01, 42)
steps = 2000
losses = array("d", bytes(8 * steps))
start = time.perf_counter()
lib.trainer_train(trainer, address(X_train), address(y_train), len(y_train), 32, steps, address(losses))
seconds = time.perf_counter() - start
first, last = sum(losses[:50]) / 50, sum(losses[-50:]) / 50
print(f"mean loss of the first 50 steps {first:.3f}, of the last 50 {last:.3f}; {seconds / steps * 1e6:.0f} µs per step")
print(f"accuracy {accuracy(trainer, X_train, y_train):.1%} on the 300 training points, "
      f"{accuracy(trainer, X_fresh, y_fresh):.1%} on 900 fresh points")
assert last < 0.5 * first and accuracy(trainer, X_fresh, y_fresh) > 0.85, "the engine should learn the spirals"

**Expected behaviour.** The mean loss falls from about 0.76 over the first 50 steps to about 0.03 over the last 50, and the network classifies about 99% of both the training points and the fresh ones. A step on 32 points takes tens of microseconds: every operation allocates a node and its vectors on the heap, which is the price of a flexible graph. Frameworks reduce it with memory pools and by fusing operations, as `cross_entropy` does here.

### Prove that training resumes exactly

The cell saves a checkpoint, continues the original run for 300 steps, loads the checkpoint into a second trainer and runs the same 300 steps there. The losses must be identical, compared with `==` rather than a tolerance. Then it tries to load a damaged file, frees both trainers, and confirms that no node is left alive.

In [ ]:
checkpoint_path = os.path.join(BUILD_DIR, "spirals.ckpt").encode()      # c_char_p takes bytes
assert lib.trainer_save(trainer, checkpoint_path) == 0, lib.last_error().decode()

continued, resumed = array("d", bytes(8 * 300)), array("d", bytes(8 * 300))
lib.trainer_train(trainer, address(X_train), address(y_train), len(y_train), 32, 300, address(continued))
restored = lib.trainer_load(checkpoint_path)
assert restored, lib.last_error().decode()
print("restored generator state and step count:", lib.trainer_rng_state(restored), lib.trainer_steps(restored))
lib.trainer_train(restored, address(X_train), address(y_train), len(y_train), 32, 300, address(resumed))
identical = list(continued) == list(resumed)
print(f"300 losses after the checkpoint: identical = {identical}; last loss {continued[-1]:.6f} and {resumed[-1]:.6f}")

with open(os.path.join(BUILD_DIR, "not_a_checkpoint.ckpt"), "wb") as handle:
    handle.write(b"this is not a checkpoint at all")
broken = lib.trainer_load(os.path.join(BUILD_DIR, "not_a_checkpoint.ckpt").encode())
print("loading a damaged file returned", broken, "with the message:", lib.last_error().decode())

lib.trainer_free(trainer)
lib.trainer_free(restored)
print("live nodes after freeing both trainers:", lib.live_node_count())
assert identical and broken is None and lib.live_node_count() == 0

**Expected behaviour.** The restored trainer reports step 2000 and the saved generator state, the 300 losses are identical, the damaged file is refused with *not a Furina checkpoint*, and no node is left alive. `ctypes` turns the `NULL` pointer from `trainer_load` into `None`. As in notebook 1, a freed trainer's pointer must never be used again.

## 10 · 🧪 Experiments: two classic training bugs

The program below runs a small model four ways from the same start: correctly; without `zero_grad`; and resumed from a checkpoint, once with the optimiser state restored and once with Adam's moments and counter reset to zero, as happens when only the weights are saved. Predict which lines will match before you run it.

In [ ]:
BUGS_MAIN = r"""
static double step_without_zero_grad(Mlp& model, Adam& adam, const Tensor& x, const std::vector<int>& labels)
{
    Tensor loss = cross_entropy(model(x), labels);    // the bug: gradients from earlier steps are kept
    backward(loss);
    adam.step();
    return loss->data[0];
}

int main()
{
    std::vector<double> X = numbers(40, 0.3);
    std::vector<int> y, labels;
    for (int i = 0; i < 20; ++i)
        y.push_back(i % 3);
    Trainer good(2, 8, 3, 0.05, 1), forgetful(2, 8, 3, 0.05, 1);
    std::vector<double> a, b;
    for (int s = 0; s < 30; ++s) {
        a.push_back(train_step(good.model, good.adam, sample_batch(good, X.data(), y.data(), 20, 8, labels), labels));
        b.push_back(step_without_zero_grad(forgetful.model, forgetful.adam,
                                           sample_batch(forgetful, X.data(), y.data(), 20, 8, labels), labels));
    }
    std::printf("loss after 30 steps, correct:         %.4f\n", a.back());
    std::printf("loss after 30 steps, no zero_grad:    %.4f\n", b.back());

    save_checkpoint(good, PATH);
    std::unique_ptr<Trainer> exact = load_checkpoint(PATH), weights_only = load_checkpoint(PATH);
    weights_only->adam.step_count = 0;                // what is lost when only the weights are saved
    for (auto& moments : weights_only->adam.m) std::fill(moments.begin(), moments.end(), 0.0);
    for (auto& moments : weights_only->adam.v) std::fill(moments.begin(), moments.end(), 0.0);
    double gap_exact = 0.0, gap_weights = 0.0;
    for (int s = 0; s < 20; ++s) {
        double base = train_step(good.model, good.adam, sample_batch(good, X.data(), y.data(), 20, 8, labels), labels);
        double e = train_step(exact->model, exact->adam, sample_batch(*exact, X.data(), y.data(), 20, 8, labels), labels);
        double w = train_step(weights_only->model, weights_only->adam,
                              sample_batch(*weights_only, X.data(), y.data(), 20, 8, labels), labels);
        gap_exact = std::max(gap_exact, std::fabs(e - base));
        gap_weights = std::max(gap_weights, std::fabs(w - base));
    }
    std::printf("largest loss gap over 20 resumed steps, full checkpoint:   %.3g\n", gap_exact);
    std::printf("largest loss gap over 20 resumed steps, optimiser reset:  %.3g\n", gap_weights);
    return 0;
}
""".replace("PATH", c_string(os.path.join(BUILD_DIR, "experiment.ckpt")))
print(run(build(engine(("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                       ("matmul", matmul_cpp()), ("cross_entropy", cross_entropy_cpp()), ("modules", MODULES_CPP),
                       ("Adam::step", adam_step_cpp()), ("Trainer", TRAINER_CPP), ("train_step", train_step_cpp()),
                       ("checkpoints", CHECKPOINT_CPP), ("load_checkpoint", load_checkpoint_cpp()))
                + mark(CHECK_CPP + BUGS_MAIN, "experiment"), "c++")))

**What to notice.** Without `zero_grad`, every step uses the sum of all earlier gradients, so the run behaves quite differently from the correct one, usually ending at a higher loss. The fully restored run matches the uninterrupted one with a gap of exactly 0. Resetting Adam's state changes the very first resumed step: the bias correction treats it as step 1 again and moves every weight by about the learning rate, so the losses diverge from the original run. Saving only the weights is fine for inference, but not for resuming.

## Exercises

Two stretch challenges. Each has a check.

### Exercise 1: `tanh_op`, a new operation

Write `tanh_op(x)`, an operation computing $\tanh$ of every entry, with its rule: $\frac{d}{dx} \tanh x = 1 - \tanh^2 x$. The rule receives its own node, so it can read the forward result from `self.data` instead of recomputing it.

*Type it rather than pasting it: typing is how the pattern sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def tanh_op_cpp_reference():
    return r"""
Tensor tanh_op(const Tensor& x)
{
    Tensor out = result(x->rows, x->cols, {x}, "tanh");
    for (size_t i = 0; i < x->size(); ++i)
        out->data[i] = std::tanh(x->data[i]);
    if (out->requires_grad)
        out->backward_rule = [](Node& self) {
            std::vector<double>& g = grad_of(*self.parents[0]);
            for (size_t i = 0; i < g.size(); ++i)
                g[i] += self.grad[i] * (1.0 - self.data[i] * self.data[i]);
        };
    return out;
}
"""

In [ ]:
def tanh_op_cpp():
    raise NotImplementedError  # ✍️ delete this line, then write the operation below
    return r"""
Tensor tanh_op(const Tensor& x)
{
    /* ✍️ Your code here: the output, the forward loop, and a rule that reads self.data */
}
"""

In [ ]:
TANH_MAIN = r"""
int main()
{
    Tensor x = tensor(2, 3, {-3.0, -0.5, 0.0, 0.25, 1.0, 20.0}, true);
    Tensor y = tanh_op(x);
    show("tanh", y->data);
    backward(sum(mul(y, tensor(2, 3, numbers(6, 0.8)))));
    show("grad", x->grad);
    return 0;
}
"""


def tanh_program(source):
    return checked_program([("topological_order", topological_order_cpp()), ("backward", backward_cpp()),
                            ("your tanh_op", source)], TANH_MAIN)


tanh_op_cpp = practice_native("tanh_op", tanh_op_cpp, tanh_op_cpp_reference, tanh_program, language="c++")

### Exercise 2: `clip_grad_norm`

Large gradients can throw a run off course in one step. Gradient clipping rescales all gradients together when their **global norm** $\sqrt{\sum g^2}$, taken over every entry of every parameter, exceeds a limit. Write `double clip_grad_norm(const std::vector<Tensor>& params, double max_norm)`: compute the global norm over the parameters whose gradient is not empty; if it exceeds `max_norm`, multiply every gradient entry by `max_norm / norm`; return the norm measured *before* clipping.

*Type it rather than pasting it: typing is how the formula sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def clip_grad_norm_cpp_reference():
    return r"""
double clip_grad_norm(const std::vector<Tensor>& params, double max_norm)
{
    double total = 0.0;
    for (const Tensor& p : params)
        for (double g : p->grad)
            total += g * g;
    double norm = std::sqrt(total);
    if (norm > max_norm)
        for (const Tensor& p : params)
            for (double& g : p->grad)
                g *= max_norm / norm;
    return norm;
}
"""

In [ ]:
def clip_grad_norm_cpp():
    raise NotImplementedError  # ✍️ delete this line, then write the function below
    return r"""
double clip_grad_norm(const std::vector<Tensor>& params, double max_norm)
{
    /* ✍️ Your code here (about ten lines) */
}
"""

In [ ]:
CLIP_MAIN = r"""
int main()
{
    Tensor a = tensor(1, 2, {0, 0}, true), b = tensor(1, 1, {0}, true), c = tensor(1, 2, {0, 0}, true);
    a->grad = {3.0, 0.0};
    b->grad = {4.0};                                   // c has no gradient yet
    double norm = clip_grad_norm({a, b, c}, 1.0);
    std::printf("norm before clipping: %.17g\n", norm);
    show("a after clipping to 1", a->grad);
    show("b after clipping to 1", b->grad);
    a->grad = {0.3, 0.0};
    b->grad = {0.4};
    norm = clip_grad_norm({a, b, c}, 1.0);             // already below the limit: unchanged
    std::printf("small norm: %.17g\n", norm);
    show("a when small", a->grad);
    return 0;
}
"""


def clip_program(source):
    return checked_program([("your clip_grad_norm", source)], CLIP_MAIN)


clip_grad_norm_cpp = practice_native("clip_grad_norm", clip_grad_norm_cpp, clip_grad_norm_cpp_reference,
                                     clip_program, language="c++")

## Common errors and debugging

- **A wall of template errors.** Read the first `error:` line only, and look at the line of your code it names. Common causes are a missing semicolon after a lambda (`};`), `->` and `.` confused (`a->data` through a pointer or `Tensor`, `node.data` through a reference), and a `const` object passed where a modifiable one is needed.
- **`error: 'x' is not captured`.** A lambda used a local variable it did not list in its capture brackets. Capture sizes by value, `[n, k, m]`; a rule gets everything else from the node it receives.
- **A crash or garbage values inside a rule.** The rule captured a local by reference (`[&]`) and ran after the local was destroyed. Capture by value or move the value in, as `cross_entropy` does with its probabilities.
- **Gradients that are too large, or that grow from step to step.** `zero_grad` is missing before `backward`, or `backward` does not reset intermediate gradients.
- **`live_nodes` keeps growing.** A rule captured a `shared_ptr` to its own node, a cycle that never frees. Rules must use the node passed to them.
- **A crash in a rule with a constant input.** The rule wrote into a parent that does not require a gradient. Test `requires_grad` before calling `grad_of` on a parent.
- **Resumed training differs from the original run.** Some state was not saved or not restored: Adam's moments and counter, or the generator's state. Compare the loss sequences with `==`, not with a tolerance.
- **`ctypes.ArgumentError` for a path.** `c_char_p` expects `bytes`, such as `path.encode()`, not a `str`.
- **The Python process exits with "terminate called after throwing an instance of …".** A C++ exception reached a function called from Python. Every `extern "C"` function must catch exceptions, as `guarded` does.

## Summary

- RAII ties resources to objects: `std::vector` frees its memory, a `unique_ptr` with `fclose` closes its file, and a `shared_ptr` destroys its object when the last owner goes away. A graph whose nodes own their parents therefore frees itself, unless a node owns itself through a cycle.
- An autograd engine records each operation's parents and a backward rule. `backward` resets intermediate gradients, seeds the root with ones and runs the rules in reverse topological order; leaf gradients accumulate until `zero_grad`.
- A rule adds local derivative times upstream gradient into each parent that needs a gradient. Fusing log-softmax and cross-entropy keeps the loss stable and the graph small.
- Adam needs its moments and step counter, and a mini-batch run needs its generator state: a checkpoint with all of them, validated by magic, version and length when loaded, resumes training exactly.
- C++ reaches Python through `extern "C"` functions that take opaque pointers and never let exceptions escape.

Notebook 3 moves the arithmetic to a graphics processor: the same matrix product and loss, written as CUDA kernels that run thousands of threads at once.